# 🏗️ Starter — LM Mini dari Nol (Bab 6)> Notebook eksperimen. **Prasyarat:** semua 40 test hijau> (`python -m unittest discover -s tests -v`).**Aturan main:** semua keputusan (dim, window, k_neg, temperature, hidden) diambildi data training/val. Angka akhir di Bagian 7 dihitung SEKALI.

## Setup & Peta Data

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

from lm_mini.corpus import KALIMAT_LM, KAT_OF, NEG, POS, TOPIK, muat_korpus_topik
from lm_mini.embeddings import frekuensi_unigram, pasangan_skipgram, sigmoid, train_skipgram
from lm_mini.languagemodel import bigram_matrix, ee_bigram, prob_baris_dengan_temp, sample_bigram
from lm_mini.rnn import rnn_backward, rnn_forward
from lm_mini.text import bangun_vocab, encode_aman, pad_sequences, tokenize_kata

korpus = muat_korpus_topik()
vocab, w2i, i2w = bangun_vocab([' '.join(s) for s in korpus])
pairs = pasangan_skipgram(korpus, w2i, window=2)
freq_uni = frekuensi_unigram(korpus, w2i)
print(f'korpus: {len(korpus)} kalimat | vocab: {len(vocab)} | pasangan: {len(pairs)}')


In [ ]:
# Lihat dulu bentuk korpusnya
print('contoh kalimat topik:', [' '.join(s) for s in korpus[:3]])
print('contoh kalimat LM  :', KALIMAT_LM[:2])
print('kategori kata      :', {w: KAT_OF[w] for w in ['kucing', 'nasi', 'bagus', 'jelek', 'itu']})


## Bagian 1 — Skip-gram: Loss Turun = Pasangan Asli Menarik, Noise DidorongLatih embedding (fungsi yang lolos gradient check di test suite).Loss per pasangan harus turun — kalau tidak, gradien-mu salah arah.

In [ ]:
E, sg_loss = train_skipgram(pairs, len(vocab), freq_uni, dim=16, epochs=5, seed=0)
print('loss:', [f'{x:.3f}' for x in sg_loss])
assert sg_loss[-1] < sg_loss[0], 'loss harus turun'

plt.plot(sg_loss, marker='o')
plt.xlabel('epoch'); plt.ylabel('loss rata-rata per pasangan')
plt.title('Skip-gram + negative sampling')
plt.show()


## Bagian 2 — Bukti Makna: Purity + PCA 2D

In [ ]:
def cos_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

cats = {'hewan', 'kendaraan', 'makanan'}
benar = total = 0
for i, w in enumerate(vocab):
    if KAT_OF[w] not in cats:
        continue
    total += 1
    bj = max((cos_sim(E[i], E[j]), j) for j in range(len(vocab)) if j != i)[1]
    benar += KAT_OF[i2w[bj]] == KAT_OF[w]
print(f'NN purity: {benar / total:.2f}  ({benar}/{total})')
assert benar / total >= 0.9, 'tetangga terdekat harus satu kategori'
print('✅ Tetangga terdekat satu kategori — TANPA pernah diberi tahu label.')


In [ ]:
def pca_2d(X):
    """PCA manual ke 2 dimensi (dari lab Bagian 5)."""
    mu = X.mean(0)
    Xc = X - mu
    Cm = (Xc.T @ Xc) / len(X)
    evals, evecs = np.linalg.eigh(Cm)
    return Xc @ evecs[:, ::-1][:, :2]

P2 = pca_2d(E)
colors = {'hewan': 'tab:red', 'kendaraan': 'tab:blue', 'makanan': 'tab:green',
          'sifat_pos': 'tab:orange', 'sifat_neg': 'tab:purple', 'fungsional': 'gray'}
plt.figure(figsize=(9, 7))
for i, w in enumerate(vocab):
    plt.scatter(P2[i, 0], P2[i, 1], c=colors[KAT_OF[w]], s=110)
    plt.annotate(w, (P2[i, 0], P2[i, 1]), xytext=(4, 4), textcoords='offset points')
plt.title('Embedding skip-gram: 3 gugus topik + kutub sentimen — tanpa label saat training')
plt.grid(alpha=0.3)
plt.show()


## Bagian 3 — Sentimen: Embedding vs Random vs One-HotFitur = rata-rata embedding kata (cara paling polos) + regresi logistik (Bab 3).

In [ ]:
def fitur_avg(toks, E_):
    return E_[[w2i[t] for t in toks]].mean(0)

def fitur_onehot_mean(toks):
    v = np.zeros(len(vocab))
    for t in toks:
        v[w2i[t]] += 1.0
    return v / len(toks)

def label_sentimen(toks):
    p = sum(t in POS for t in toks)
    n = sum(t in NEG for t in toks)
    return 1 if p > n else (-1 if n > p else 0)

def latih_logreg(X, y, epochs=300, lr=0.5, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_tr = int(0.75 * len(X))
    Xtr, ytr, Xva, yva = X[idx[:n_tr]], y[idx[:n_tr]], X[idx[n_tr:]], y[idx[n_tr:]]
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(epochs):
        p = sigmoid(Xtr @ w + b)
        w -= lr * (Xtr.T @ (p - ytr) / len(ytr))
        b -= lr * (p - ytr).mean()
    pv = sigmoid(Xva @ w + b)
    return float(((pv >= 0.5) == yva).mean())

data = [(s, 1.0 if label_sentimen(s) == 1 else 0.0) for s in korpus if label_sentimen(s) != 0]
y_bin = np.array([l for _, l in data])
E_rand = np.random.default_rng(7).normal(0, 0.1, (len(vocab), E.shape[1]))
X_oh = np.array([fitur_onehot_mean(s) for s, _ in data])
X_rd = np.array([fitur_avg(s, E_rand) for s, _ in data])
X_sg = np.array([fitur_avg(s, E) for s, _ in data])
acc_oh = latih_logreg(X_oh, y_bin)
acc_rd = latih_logreg(X_rd, y_bin)
acc_sg = latih_logreg(X_sg, y_bin)
print(f'one-hot mean : {acc_oh:.3f}')
print(f'random emb   : {acc_rd:.3f}')
print(f'skip-gram emb: {acc_sg:.3f}')
assert acc_sg > acc_rd + 0.15, 'skip-gram embedding harus mengalahkan random'
print('✅ Embedding bermakna — makna yang membuat fitur generalisasi.')


## Bagian 4 — Char-LM Bigram: Cross-Entropy & TemperatureBaseline model bahasa: hitung, evaluasi EE, generate di 3 temperature.

In [ ]:
teks_lm = '\n'.join(KALIMAT_LM) + '\n'
chars = sorted(set(teks_lm))
c2i = {c: i for i, c in enumerate(chars)}
C = len(chars)
seq_lm = [c2i[c] for c in teks_lm]
P_bigram = bigram_matrix(seq_lm, C, alpha=0.5)
ee_b = ee_bigram(seq_lm, P_bigram)
unigram = np.bincount(seq_lm, minlength=C) / len(seq_lm)
ee_u = -np.mean(np.log(unigram[seq_lm[1:]] + 1e-12))
print(f'EE unigram {ee_u:.3f} vs bigram {ee_b:.3f} nats/char')

rng_g = np.random.default_rng(1)
for t in (0.3, 1.0, 1.8):
    ids = sample_bigram(P_bigram, c2i['m'], 90, t, rng_g)
    print(f'--- T={t} ---')
    print(''.join(chars[i] for i in ids))

def entropy_sampel(id_sampel):
    cnts = Counter(''.join(chars[i] for i in id_sampel))
    p = np.array(list(cnts.values())) / sum(cnts.values())
    return float(-(p * np.log(p)).sum())

Tk = [sample_bigram(P_bigram, c2i['m'], 80, 0.3, rng_g) for _ in range(20)]
Tb = [sample_bigram(P_bigram, c2i['m'], 80, 1.8, rng_g) for _ in range(20)]
print(f"entropy karakter: T=0.3 {np.mean([entropy_sampel(s) for s in Tk]):.3f}"
      f" | T=1.8 {np.mean([entropy_sampel(s) for s in Tb]):.3f}")
assert np.mean([entropy_sampel(s) for s in Tb]) > np.mean([entropy_sampel(s) for s in Tk])
print('✅ Temperature berperilaku benar: T rendah repetitif, T tinggi beragam.')


## Bagian 5 — RNN Char-Level vs Bigram

In [ ]:
CTX = 4
X_rnn, y_rnn = [], []
for i in range(len(seq_lm) - CTX):
    X_rnn.append(seq_lm[i:i + CTX])
    y_rnn.append(seq_lm[i + CTX])
X_rnn, y_rnn = np.array(X_rnn), np.array(y_rnn)
perm = np.random.default_rng(0).permutation(len(X_rnn))
X_rnn, y_rnn = X_rnn[perm], y_rnn[perm]
n_tr = int(0.75 * len(X_rnn))
Xtr, ytr, Xva, yva = X_rnn[:n_tr], y_rnn[:n_tr], X_rnn[n_tr:], y_rnn[n_tr:]

hidden, epochs, lr = 32, 150, 0.2
rng = np.random.default_rng(0)
params = [rng.normal(0, 0.5, (C, hidden)), rng.normal(0, 0.5, (hidden, hidden)),
          rng.normal(0, 0.1, (hidden, C)), np.zeros(hidden), np.zeros(C)]
hist = []
for ep in range(epochs):
    order = rng.permutation(len(Xtr))
    tot, cnt = 0.0, 0
    for s in range(0, len(order), 32):
        bidx = order[s:s + 32]
        Xb, yb = Xtr[bidx], ytr[bidx]
        probs, cache = rnn_forward(Xb, params, CTX)
        n = len(bidx)
        tot += -np.mean(np.log(probs[np.arange(n), yb] + 1e-12)); cnt += 1
        grads = rnn_backward(probs, yb, Xb, params, cache, CTX)
        for g in grads:
            np.clip(g, -5, 5, out=g)
        for P, G in zip(params, grads):
            P -= lr * G
    hist.append(tot / cnt)
probs_va, _ = rnn_forward(Xva, params, CTX)
acc_rnn = float((probs_va.argmax(1) == yva).mean())
counts_b = np.ones((C, C))
for a, b in zip(seq_lm[:n_tr + CTX], seq_lm[1:n_tr + CTX]):
    counts_b[a, b] += 1
P_tr = counts_b / counts_b.sum(1, keepdims=True)
acc_bi = float((np.array([P_tr[x[-1]].argmax() for x in Xva]) == yva).mean())
print(f'RNN   : val acc {acc_rnn:.3f} | loss {hist[0]:.3f} -> {hist[-1]:.3f}')
print(f'bigram: val acc {acc_bi:.3f}')
assert acc_rnn > acc_bi, f'RNN harus menang: {acc_rnn:.3f} vs {acc_bi:.3f}'
print('✅ Hidden state (konteks penuh) mengalahkan bigram (1 char).')

plt.plot(hist, marker='o')
plt.xlabel('epoch'); plt.ylabel('CE train')
plt.title('RNN char-level: loss turun = hidden state menangkap pola')
plt.show()


## Bagian 6 — Vanishing Gradient (kenapa konteks jauh "tidak pernah belajar")

In [ ]:
def bptt_norm(act, T=30, seed=0, radius=0.9):
    """||gradien|| saat mengalir ke belakang T langkah (dari lab Bagian 9)."""
    rng = np.random.default_rng(seed)
    Wh = np.abs(rng.normal(0, 0.35, (20, 20)))
    Wh = Wh / np.linalg.norm(Wh, 2) * radius
    g = np.full(20, 0.5)
    out = []
    for _ in range(T):
        if act == 'tanh':
            deriv = 1 - np.tanh(np.linspace(-1.5, 1.5, 20)) ** 2
        else:
            s = 1 / (1 + np.exp(-np.linspace(-2.5, 2.5, 20)))
            deriv = s * (1 - s)
        g = Wh @ (g * deriv)
        out.append(float(g.mean()))
    return out

plt.semilogy(bptt_norm('tanh'), label='tanh')
plt.semilogy(bptt_norm('sigmoid'), label='sigmoid')
plt.xlabel('langkah waktu ke belakang'); plt.ylabel('||gradien|| rata-rata')
plt.title('Vanishing gradient — alasan lahirnya LSTM & Transformer')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"T=30: tanh {bptt_norm('tanh')[-1]:.1e} vs sigmoid {bptt_norm('sigmoid')[-1]:.1e}")


## Bagian 7 — Ringkasan Angka Akhir (SEKALI saja)Isi RUBRIK.md dari angka-angka ini. Setelah cell ini dijalankan, eksperimen selesai.

In [ ]:
def nn_purity(E_):
    def cos(a, b):
        return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
    cats = {'hewan', 'kendaraan', 'makanan'}
    benar = total = 0
    for i, w in enumerate(vocab):
        if KAT_OF[w] not in cats:
            continue
        total += 1
        bj = max((cos(E_[i], E_[j]), j) for j in range(len(vocab)) if j != i)[1]
        benar += KAT_OF[i2w[bj]] == KAT_OF[w]
    return benar / total

print('=' * 52)
print('RINGKASAN ANGKA AKHIR (isi RUBRIK.md dari sini)')
print('=' * 52)
print(f'1. purity embedding        : {nn_purity(E):.2f}')
print(f'2. akurasi sentimen SG     : {acc_sg:.3f} (one-hot {acc_oh:.3f}, random {acc_rd:.3f})')
print(f'3. EE bigram vs unigram    : {ee_b:.3f} vs {ee_u:.3f} nats/char')
print(f'4. val acc RNN vs bigram   : {acc_rnn:.3f} vs {acc_bi:.3f}')
print('Test/val disentuh sekali — angka ini final. Tidak ada "coba-coba lagi".')


**Pertanyaan Analisis** (jawab di cell markdown — bagian penilaian, lihat RUBRIK.md):
1. Mekanisme ko-okurensi: kenapa 'kucing'–'anjing' berdekatan tapi 'kucing'–'nasi' tidak?2. Apa yang hilang tanpa negative sampling? (eksperimen kecil di bawah)3. Kenapa embedding generalisasi tapi one-hot menghafal? Kaitkan dengan angka akurasi-mu.4. Temperature: tempel dua sampel generasi-mu (T rendah vs tinggi) + kapan memilih yang mana.5. Apa yang sebenarnya dipelajari hidden state RNN? Apa bukti konteks 1 char tidak cukup?6. Vanishing gradient: kenapa gradien menurun sepanjang waktu + dua solusi arsitektural.

### (Opsional) Eksperimen untuk Pertanyaan 2 — tanpa negative samplingSalin `train_skipgram` ke sini, hapus bagian negative sampling (hanya pasanganasli), latih ulang, bandingkan purity. Apa yang terjadi pada geometri embedding?

In [ ]:
# TODO: latih versi TANPA negative sampling di sini, lalu bandingkan purity
# petunjuk: gradien hanya dari pasangan asli: grad_v = (s-1)*u, grad_u = (s-1)*v
